# 02 – Data Preparation

Apply the treatments identified in the EDA, without leaking information from the test set.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/cs-training.csv")
df = df.drop(columns=["Unnamed: 0"])

# Anomalía 1: edad 0 (un solo registro inválido)
df = df[df["age"] > 0]

print(df.shape)

(149999, 11)


## 1. Flags
Binary indicators created *before* correcting any value, so the information is not lost.

In [2]:
df["income_missing"] = df["MonthlyIncome"].isnull().astype(int)
df["dependents_missing"] = df["NumberOfDependents"].isnull().astype(int)
df["special_code"] = (df["NumberOfTimes90DaysLate"] >= 90).astype(int)
df["util_invalid"] = (df["RevolvingUtilizationOfUnsecuredLines"] > 10).astype(int)

flags = ["income_missing", "dependents_missing", "special_code", "util_invalid"]
df[flags].sum()

income_missing        29731
dependents_missing     3924
special_code            269
util_invalid            241
dtype: int64

## 2. Fix invalid values
- Codes 96/98 in past-due counts → 0 (the `special_code` flag keeps the information).
- Revolving utilization > 10 → missing (imputed later with the train median).
- DebtRatio when income is missing → missing (it is an amount, not a ratio).

In [3]:
mora = ["NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate"]

# Códigos 96/98 -> 0
df.loc[df["special_code"] == 1, mora] = 0

# Utilización inválida -> faltante
df.loc[df["util_invalid"] == 1, "RevolvingUtilizationOfUnsecuredLines"] = np.nan

# DebtRatio sin ingreso -> faltante
df.loc[df["income_missing"] == 1, "DebtRatio"] = np.nan

# Verificación
print("Max past-due counts:", df[mora].max().tolist())
print("Max utilization:", df["RevolvingUtilizationOfUnsecuredLines"].max())
print(df.isnull().sum()[df.isnull().sum() > 0])

Max past-due counts: [13, 11, 17]
Max utilization: 8.851851852
RevolvingUtilizationOfUnsecuredLines      241
DebtRatio                               29731
MonthlyIncome                           29731
NumberOfDependents                       3924
dtype: int64


## 3. Train / test split
Same split as in the thesis (80/20, stratified, seed 42), so results are comparable.

In [4]:
X = df.drop(columns="SeriousDlqin2yrs")
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, f"default rate {y_train.mean():.2%}")
print("Test: ", X_test.shape, f"default rate {y_test.mean():.2%}")

Train: (119999, 14) default rate 6.68%
Test:  (30000, 14) default rate 6.68%


## 4. Imputation and capping (fitted on train only)
Medians and 99th percentiles are computed on the training set and applied to both sets.

In [5]:
# 1) Rellenar faltantes con la mediana del TRAIN
a_imputar = ["MonthlyIncome", "NumberOfDependents",
             "RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]

medianas = X_train[a_imputar].median()
X_train = X_train.fillna(medianas)
X_test = X_test.fillna(medianas)

# 2) Acotar valores extremos al percentil 99 del TRAIN
a_acotar = ["DebtRatio", "MonthlyIncome"]

topes = X_train[a_acotar].quantile(0.99)
X_train[a_acotar] = X_train[a_acotar].clip(upper=topes, axis=1)
X_test[a_acotar] = X_test[a_acotar].clip(upper=topes, axis=1)

print("Medians (train):\n", medianas.round(2), "\n")
print("Caps (train p99):\n", topes.round(2), "\n")
print("Missing left:", X_train.isnull().sum().sum(), "/", X_test.isnull().sum().sum())

Medians (train):
 MonthlyIncome                           5400.00
NumberOfDependents                         0.00
RevolvingUtilizationOfUnsecuredLines       0.15
DebtRatio                                  0.30
dtype: float64 

Caps (train p99):
 DebtRatio          266.06
MonthlyIncome    23018.30
Name: 0.99, dtype: float64 

Missing left: 0 / 0


## 5. Save processed data

In [6]:
train = X_train.copy()
train["SeriousDlqin2yrs"] = y_train

test = X_test.copy()
test["SeriousDlqin2yrs"] = y_test

train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

print("Saved:", train.shape, test.shape)

Saved: (119999, 15) (30000, 15)


## Summary

| Step | Treatment | Fitted on |
|---|---|---|
| Invalid age | Dropped 1 borrower with `age` = 0 | Rule |
| Flags | `income_missing`, `dependents_missing`, `special_code`, `util_invalid` created before any correction | Rule |
| Codes 96/98 | Replaced with 0 in the three past-due variables (info kept in `special_code`) | Rule |
| Utilization > 10 | Set to missing, then imputed | Rule + train median |
| DebtRatio without income | Set to missing, then imputed (it was an amount, not a ratio) | Rule + train median |
| Missing values | Imputed with medians | Train only |
| Extreme values | `DebtRatio` and `MonthlyIncome` capped at the 99th percentile | Train only |

**Output:** `data/processed/train.csv` (119,999 rows) and `test.csv` (30,000 rows), 14 features + target, no missing values. Default rate 6.68% in both.

**Difference vs. the thesis:** the riskiest borrowers (codes 96/98, 54.65% default rate) are kept instead of dropped, missingness is preserved as a feature, and no statistic is computed on the test set.

**Open point:** `DebtRatio` still reaches 266 after capping, likely borrowers reporting near-zero income. To review.